## 1. Import libraries and configure the inference environment

In [ ]:
from    typing  import  Sequence
from    itertools   import  product

from    tqdm.notebook   import  tqdm
from    time            import  perf_counter

import  torch
from    torch.utils.data                import  DataLoader

from    deep_numerical.numerical.solver import  FastSM_Boltzmann_VHS, one_step_RK4_classic, LAMBDA
from    deep_numerical.numerical.math   import  area_of_unit_sphere

from    deep_numerical.utils            import  GaussianNormalizer, relative_error, sec_to_hms
from    deep_numerical.neural.utils     import  count_parameters
from    deep_numerical.neural.operator  import  ModifiedFNO, PFNO, ModifiedRFNO, PRFNO
import  matplotlib.pyplot               as  plt

from    loader      import  NormKeys, HistKeys
from    loader      import  path_base
from    loader      import  load_model_args, load_model_data
from    dataset     import  nonparametric_dataset, parametric_dataset
from    plot        import  *


DEVICE: torch.device    = torch.device("cuda:1")
DIMENSION:  int         = 2
MAX_V:      float       = 3.0/LAMBDA
VHS_COEFF:  float       = 1.0/area_of_unit_sphere(DIMENSION)

list_max_freq   = [6, 8, 10]
list_seed       = [0, 1, 2]
max_freq        = list_max_freq[0]
seed            = list_seed[2]

res_v = 2**6

## 2. Instantiate the trained operators and normalizers

### 2-1. Load data (trained parameters, normalizers, and training histories)

In [2]:
model_data_fno      = load_model_data("FNO2D-param", max_freq, seed)
model_data_pfno     = load_model_data("PFNO2D-param", max_freq, seed)
model_data_rfno     = load_model_data("RFNO2D-param", max_freq, seed)
model_data_prfno    = load_model_data("PRFNO2D-param", max_freq, seed)

### 2-2. Load normalizers

In [3]:
norm_fno:   dict[NormKeys, GaussianNormalizer] = model_data_fno["normalizer"]
norm_pfno:  dict[NormKeys, GaussianNormalizer] = model_data_pfno["normalizer"]
norm_rfno:  dict[NormKeys, GaussianNormalizer] = model_data_rfno["normalizer"]
norm_prfno: dict[NormKeys, GaussianNormalizer] = model_data_prfno["normalizer"]
for v in norm_fno.values():   v.to(DEVICE)
for v in norm_pfno.values():  v.to(DEVICE)
for v in norm_rfno.values():  v.to(DEVICE)
for v in norm_prfno.values(): v.to(DEVICE)
print(f"Successfully loaded normalizers.")

Successfully loaded normalizers.


### 2-3. Load trained operators

In [4]:
model_fno   = ModifiedFNO(**load_model_args("FNO2D-param", max_freq))
model_pfno  = PFNO(**load_model_args("PFNO2D-param", max_freq))
model_rfno  = ModifiedRFNO(**load_model_args("RFNO2D-param", max_freq))
model_prfno = PRFNO(**load_model_args("PRFNO2D-param", max_freq))

model_fno.load_state_dict(  model_data_fno["state_dict"])
model_pfno.load_state_dict( model_data_pfno["state_dict"])
model_rfno.load_state_dict( model_data_rfno["state_dict"])
model_prfno.load_state_dict(model_data_prfno["state_dict"])
model_fno.to(DEVICE)
model_pfno.to(DEVICE)
model_rfno.to(DEVICE)
model_prfno.to(DEVICE)
model_fno.eval()
model_pfno.eval()
model_rfno.eval()
model_prfno.eval()
print(f"Successfully loaded models.")

Successfully loaded models.


## 3. Conduct inference

### 3-1. Construct dataloader

In [5]:
data, params = [], []
# list_alpha = torch.arange(21) * 0.05
# for alpha in list_alpha:
alpha = 1.00
file_name = f"ood_test_dataset/res{res_v:03d}/FSM_Boltzmann__VHS__distribution__bimaxwellian__dim2__res{res_v}__alpha{alpha:0.2f}__inst1.pth"
full_trajectory: torch.Tensor = torch.load(file_name, weights_only=False)
data.append(full_trajectory[:, 0].squeeze().unsqueeze(-1))
params.append(alpha*torch.ones(1))  # float
data    = torch.stack(data, dim=0).to(DEVICE)
params  = torch.stack(params, dim=0).to(DEVICE)


data    = data[:, None]
data    = data.type(torch.float)
params  = params.type(torch.float)


min_t = 0.0
max_t = 5.0
delta_t = 0.1
res_t = int((max_t - min_t) / delta_t + 1.01)


num_inst = data.size(0)
res_v = data.size(-2)

print(data.shape, params.shape)
print(f"Successfully loaded dataset with {num_inst} instances.")

torch.Size([1, 1, 128, 128, 1]) torch.Size([1, 1])
Successfully loaded dataset with 1 instances.


In [6]:
BATCH_SIZE = 2**6

dataloader_fno  = DataLoader(nonparametric_dataset(data, params, norm_fno), batch_size=BATCH_SIZE, shuffle=False)
dataloader_pfno = DataLoader(parametric_dataset(data, params, norm_pfno), batch_size=BATCH_SIZE, shuffle=False)
dataloader_rfno = DataLoader(nonparametric_dataset(data, params, norm_rfno), batch_size=BATCH_SIZE, shuffle=False)
dataloader_prfno = DataLoader(parametric_dataset(data, params, norm_prfno), batch_size=BATCH_SIZE, shuffle=False)
print(f"Successfully prepared dataloaders.")

Successfully prepared dataloaders.


### 3-2. Compute the trajectories on the test set

Fast spectral method

In [ ]:
%%timeit -r 10 -n 5
fsm = FastSM_Boltzmann_VHS(DIMENSION, res_v, MAX_V, vhs_coeff=VHS_COEFF, vhs_alpha=alpha, dtype=torch.float32, device=DEVICE)
arr_f = data.reshape(1, *(1 for _ in range(DIMENSION)), *(res_v for _ in range(DIMENSION)), 1) # (batch, x, v, channel)

inference_time_fsm = perf_counter()
with torch.inference_mode():
    pred_fsm = fsm.solve(min_t, max_t, delta_t, arr_f, one_step_RK4_classic)
inference_time_fsm = perf_counter() - inference_time_fsm
pred_fsm = pred_fsm.reshape(num_inst, res_t, *(res_v for _ in range(DIMENSION)), 1).cpu()
# print(f"FSM inference time: {inference_time_fsm:.2f} seconds.")

FSM inference time: 5.78 seconds.


FNO

In [8]:
# NOTE: The first measurement might be incorrect,
# so the inference of FNO is conducted twice, where the first inference is for warm-up.
pred_fno = []

with torch.inference_mode():
    model_preds = []
    traj:   torch.Tensor
    alphas: torch.Tensor
    for traj, alphas in tqdm(dataloader_fno, desc="FNO Inference"):
        traj    = traj.to(DEVICE)
        alphas  = alphas.to(DEVICE)
        pred    = traj[:, 0]    # The initial condition of shape `(batch_size, res, res, 1)`
        # Save the initial condition
        batch_pred = [norm_fno['data'].decode(pred).cpu()]
        # Compute the trajectory
        for t in range(1, res_t):
            pred = model_fno.forward(pred, alphas)
            batch_pred.append(norm_fno['data'].decode(pred).cpu())
        batch_pred = torch.stack(batch_pred, dim=1)   # (batch_size, res_t, res, res, 1)
        model_preds.append(batch_pred)
    pred_fno = torch.cat(model_preds, dim=0).cpu()   # (num_inst, res_t, res, res, 1)
print("Warm-up inference of FNO done.")

FNO Inference:   0%|          | 0/1 [00:00<?, ?it/s]

Warm-up inference of FNO done.


In [ ]:
%%timeit -r 10 -n 5
# NOTE: The first measurement might be incorrect,
# so the inference of FNO is conducted twice, where the first inference is for warm-up.
pred_fno = []
inference_time_fno = perf_counter()

with torch.inference_mode():
    model_preds = []
    traj:   torch.Tensor
    alphas: torch.Tensor
    for traj, alphas in tqdm(dataloader_fno, desc="FNO Inference"):
        traj    = traj.to(DEVICE)
        alphas  = alphas.to(DEVICE)
        pred    = traj[:, 0]    # The initial condition of shape `(batch_size, res, res, 1)`
        # Save the initial condition
        batch_pred = [norm_fno['data'].decode(pred).cpu()]
        # Compute the trajectory
        for t in range(1, res_t):
            pred = model_fno.forward(pred, alphas)
            batch_pred.append(norm_fno['data'].decode(pred).cpu())
        batch_pred = torch.stack(batch_pred, dim=1)   # (batch_size, res_t, res, res, 1)
        model_preds.append(batch_pred)
    pred_fno = torch.cat(model_preds, dim=0).cpu()   # (num_inst, res_t, res, res, 1)

inference_time_fno = perf_counter() - inference_time_fno
# print(f"FNO Inference time: {inference_time_fno:.2f} seconds.")

FNO Inference:   0%|          | 0/1 [00:00<?, ?it/s]

FNO Inference:   0%|          | 0/1 [00:00<?, ?it/s]

FNO Inference:   0%|          | 0/1 [00:00<?, ?it/s]

FNO Inference:   0%|          | 0/1 [00:00<?, ?it/s]

FNO Inference:   0%|          | 0/1 [00:00<?, ?it/s]

FNO Inference:   0%|          | 0/1 [00:00<?, ?it/s]

FNO Inference:   0%|          | 0/1 [00:00<?, ?it/s]

FNO Inference:   0%|          | 0/1 [00:00<?, ?it/s]

FNO Inference:   0%|          | 0/1 [00:00<?, ?it/s]

PFNO

In [ ]:
%%timeit -r 10 -n 5
pred_pfno = []
inference_time_pfno = perf_counter()

with torch.inference_mode():
    model_preds = []
    traj:   torch.Tensor
    alphas: torch.Tensor
    for traj, alphas in tqdm(dataloader_pfno, desc="PFNO Inference"):
        traj    = traj.to(DEVICE)
        alphas  = alphas.to(DEVICE)
        pred    = traj[:, 0]    # The initial condition of shape `(batch_size, res, res, 1)`
        # Save the initial condition
        batch_pred = [norm_pfno['data'].decode(pred).cpu()]
        # Compute the trajectory
        for t in range(1, res_t):
            pred = model_pfno.forward(pred, alphas)
            batch_pred.append(norm_pfno['data'].decode(pred).cpu())
        batch_pred = torch.stack(batch_pred, dim=1)   # (batch_size, res_t, res, res, 1)
        model_preds.append(batch_pred)
    pred_pfno = torch.cat(model_preds, dim=0).cpu()   # (num_inst, res_t, res, res, 1)

inference_time_pfno = perf_counter() - inference_time_pfno
# print(f"PFNO Inference time: {inference_time_pfno:.2f} seconds.")

RFNO

In [ ]:
%%timeit -r 10 -n 5
pred_rfno = []
inference_time_rfno = perf_counter()

with torch.inference_mode():
    model_preds = []
    traj:   torch.Tensor
    alphas: torch.Tensor
    for traj, alphas in tqdm(dataloader_rfno, desc="RFNO Inference"):
        traj    = traj.to(DEVICE)
        alphas  = alphas.to(DEVICE)
        pred    = traj[:, 0]    # The initial condition of shape `(batch_size, res, res, 1)`
        # Save the initial condition
        batch_pred = [norm_rfno['data'].decode(pred).cpu()]
        # Compute the trajectory
        for t in range(1, res_t):
            pred = model_rfno.forward(pred, alphas)
            batch_pred.append(norm_rfno['data'].decode(pred).cpu())
        batch_pred = torch.stack(batch_pred, dim=1)   # (batch_size, res_t, res, res, 1)
        model_preds.append(batch_pred)
    pred_rfno = torch.cat(model_preds, dim=0).cpu()   # (num_inst, res_t, res, res, 1)

inference_time_rfno = perf_counter() - inference_time_rfno
# print(f"RFNO Inference time: {inference_time_rfno:.2f} seconds.")

PRFNO

In [ ]:
%%timeit -r 10 -n 5
pred_prfno = []
inference_time_prfno = perf_counter()

with torch.inference_mode():
    model_preds = []
    traj:   torch.Tensor
    alphas: torch.Tensor
    for traj, alphas in tqdm(dataloader_prfno, desc="PRFNO Inference"):
        traj    = traj.to(DEVICE)
        alphas  = alphas.to(DEVICE)
        pred    = traj[:, 0]    # The initial condition of shape `(batch_size, res, res, 1)`
        # Save the initial condition
        batch_pred = [norm_prfno['data'].decode(pred).cpu()]
        # Compute the trajectory
        for t in range(1, res_t):
            pred = model_prfno.forward(pred, alphas)
            batch_pred.append(norm_prfno['data'].decode(pred).cpu())
        batch_pred = torch.stack(batch_pred, dim=1)   # (batch_size, res_t, res, res, 1)
        model_preds.append(batch_pred)
    pred_prfno = torch.cat(model_preds, dim=0).cpu()   # (num_inst, res_t, res, res, 1)

inference_time_prfno = perf_counter() - inference_time_prfno
# print(f"PRFNO Inference time: {inference_time_prfno:.2f} seconds.")

End of file